# Sprint 1: CIERRE

Este notebook completa el Sprint 1:

1. **Evaluacion sobre test** (datos NO vistos durante entrenamiento).
2. **Curva ROC + matriz de confusion** sobre test.
3. **Grad-CAM** sobre 6 imagenes (3 cardiomegalia + 3 normales).

Instala los 3 archivos finales en Drive:
- `src/evaluation/evaluate.py`
- `scripts/run_eval.py`
- `src/explainability/gradcam.py`

**Runtime:** T4 GPU.
**Tiempo estimado:** 2-5 min.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'

import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
!pip install -q torchxrayvision scipy grad-cam


## 2. Escribir los 3 archivos finales

In [ ]:
from pathlib import Path

# evaluate.py
content = r'''"""
src/evaluation/evaluate.py

Evaluacion final del modelo sobre el test set.

Carga el checkpoint guardado durante entrenamiento, corre inferencia sobre test,
y calcula las 5 metricas del charter + curva ROC + matriz de confusion.

Los resultados se guardan en experiments/:
  - test_metrics.json: las 5 metricas + probabilidades + labels
  - test_roc.png: curva ROC
  - test_confusion_matrix.png: matriz de confusion visualizada
"""
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch.utils.data import DataLoader

from src.evaluation.metrics import compute_metrics, logits_to_probs


@torch.no_grad()
def evaluate_model(model: torch.nn.Module, test_loader: DataLoader,
                   device: torch.device, threshold: float = 0.5,
                   logger=None) -> dict:
    """
    Corre inferencia sobre test_loader y devuelve metricas + raw predictions.

    Returns:
        {
          'metrics': ClassificationMetrics,
          'probs': np.ndarray (N,) probabilidades de clase positiva,
          'labels': np.ndarray (N,) labels verdaderos,
          'image_indices': list[str] nombres de archivos correspondientes,
        }
    """
    model.eval()

    all_probs = []
    all_labels = []
    all_names = []

    for batch in test_loader:
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label']
        names = batch['image_index']

        logits = model(images)
        probs = logits_to_probs(logits.float(), positive_class=1)

        all_probs.append(probs.cpu())
        all_labels.append(labels)
        all_names.extend(names)

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    metrics = compute_metrics(all_labels, all_probs, threshold=threshold)

    if logger is not None:
        logger.info("Evaluacion sobre test: %d muestras", len(all_labels))
        logger.info("  %s", metrics)

    return {
        'metrics': metrics,
        'probs': all_probs,
        'labels': all_labels,
        'image_indices': all_names,
    }


def save_test_results(result: dict, output_dir: Path | str) -> None:
    """Guarda test_metrics.json con los resultados de la evaluacion."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    payload = {
        'metrics': result['metrics'].as_dict(),
        'probs': result['probs'].tolist(),
        'labels': result['labels'].tolist(),
        'image_indices': list(result['image_indices']),
    }

    path = output_dir / 'test_metrics.json'
    with path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def load_model_from_checkpoint(checkpoint_path: Path | str, cfg,
                               device: torch.device) -> torch.nn.Module:
    """
    Reconstruye el modelo y carga los pesos del checkpoint.

    Importante: necesita la misma config que se uso en entrenamiento.
    """
    from src.models.baseline import build_baseline_model

    model = build_baseline_model(cfg).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device,
                            weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model
'''
path = f'{CODE_DIR}/src/evaluation/evaluate.py'
Path(path).parent.mkdir(parents=True, exist_ok=True)
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


In [ ]:
# run_eval.py
content = r'''"""
scripts/run_eval.py

Entrypoint: evalua el mejor checkpoint sobre test set.
"""
from __future__ import annotations

import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.datasets.nih import build_nih_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.evaluation.evaluate import (
    evaluate_model,
    load_model_from_checkpoint,
    save_test_results,
)
from src.training.utils import (
    get_device,
    get_logger,
    load_config,
    set_seed,
)


def main(config_path: str | None = None) -> None:
    if config_path is None:
        config_path = CODE_DIR / "configs" / "baseline.yaml"
    cfg = load_config(config_path)

    set_seed(cfg.project['seed'])

    log_file = Path(cfg.paths['logs']) / "evaluate.log"
    logger = get_logger("evaluate", log_file=log_file)

    logger.info("=" * 70)
    logger.info("RUN EVAL - Sprint 1 baseline")
    logger.info("=" * 70)

    device = get_device()
    logger.info("Device: %s", device)

    # 1. Dataset de test (con transform de evaluacion, SIN augmentation)
    train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])
    eval_tfm = build_eval_transform(image_size=cfg.preprocessing['image_size'])
    _, _, test_ds = build_nih_datasets(cfg, train_tfm, eval_tfm)

    test_loader = DataLoader(
        test_ds, batch_size=cfg.training['batch_size'], shuffle=False,
        num_workers=cfg.training.get('num_workers', 0),
        pin_memory=(device.type == 'cuda'),
    )
    logger.info("Test: %d muestras | distribucion: %s",
                len(test_ds), test_ds.class_distribution())

    # 2. Cargar checkpoint
    ckpt_path = Path(cfg.paths['checkpoints']) / 'baseline_best.pt'
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f"Checkpoint no encontrado: {ckpt_path}. "
            "Corre primero scripts/run_train.py."
        )
    logger.info("Checkpoint: %s", ckpt_path)
    model = load_model_from_checkpoint(ckpt_path, cfg, device)

    # 3. Evaluar
    threshold = cfg.evaluation.get('threshold', 0.5)
    result = evaluate_model(model, test_loader, device,
                            threshold=threshold, logger=logger)

    # 4. Guardar
    save_test_results(result, Path(cfg.paths['experiments']))
    logger.info("Resultados guardados en: %s/test_metrics.json",
                cfg.paths['experiments'])

    logger.info("=" * 70)
    logger.info("RESUMEN TEST:")
    logger.info("  %s", result['metrics'])


if __name__ == "__main__":
    main()
'''
path = f'{CODE_DIR}/scripts/run_eval.py'
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


In [ ]:
# gradcam.py
content = r'''"""
src/explainability/gradcam.py

Grad-CAM (Gradient-weighted Class Activation Mapping) para el modelo baseline.

Grad-CAM ilumina las regiones de la imagen que mas contribuyen a la prediccion
de una clase especifica. Para cardiomegalia, esperamos que el mapa de calor
ilumine la region cardiaca (silueta del corazon).

Implementacion: usamos la libreria `pytorch-grad-cam` que ya esta en requirements.

Target layer: la ultima conv del backbone DenseNet121.
  En xrv.models.DenseNet la estructura es:
    backbone.features.denseblock4.denselayer16.conv2  (ultima conv)
  Pero lo mas comun es apuntar al ultimo denseblock entero:
    backbone.features.denseblock4
"""
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


def get_target_layer(model: nn.Module, target_layer_name: str = "features.denseblock4"):
    """
    Navega la jerarquia del modelo para encontrar la capa objetivo.

    Ejemplo: "features.denseblock4" -> model.backbone.features.denseblock4
    """
    # El modelo tiene backbone adentro
    module = model.backbone
    for part in target_layer_name.split('.'):
        module = getattr(module, part)
    return module


def normalize_for_display(img: np.ndarray) -> np.ndarray:
    """
    Normaliza una imagen con rango [-1024, 1024] (formato xrv) a [0, 1]
    para poder superponerla con el mapa de calor.
    """
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img


def compute_gradcam_for_sample(model: nn.Module, image_tensor: torch.Tensor,
                                target_class: int,
                                target_layer) -> np.ndarray:
    """
    Calcula Grad-CAM para una imagen.

    Args:
        model: modelo entrenado.
        image_tensor: tensor (1, H, W) o (1, 1, H, W) en formato xrv.
        target_class: clase objetivo (0 o 1).
        target_layer: capa del modelo a visualizar.

    Returns:
        heatmap: np.ndarray (H, W) en [0, 1].
    """
    # Asegurar shape (1, 1, H, W) para batch
    if image_tensor.ndim == 3:
        input_tensor = image_tensor.unsqueeze(0)
    else:
        input_tensor = image_tensor

    input_tensor = input_tensor.requires_grad_(True)

    # pytorch-grad-cam espera que el modelo devuelva logits
    cam = GradCAM(model=model, target_layers=[target_layer])
    targets = [ClassifierOutputTarget(target_class)]

    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    return grayscale_cam[0]  # (H, W)


def overlay_heatmap(image: np.ndarray, heatmap: np.ndarray,
                    alpha: float = 0.5) -> np.ndarray:
    """
    Superpone el heatmap sobre la imagen en escala de grises.

    Args:
        image: (H, W) en [0, 1] (imagen base).
        heatmap: (H, W) en [0, 1] (Grad-CAM).
        alpha: peso del heatmap (0=solo imagen, 1=solo heatmap).

    Returns:
        RGB image (H, W, 3) en [0, 1].
    """
    # Imagen base en escala de grises repetida a 3 canales
    img_rgb = np.stack([image] * 3, axis=-1)

    # Heatmap en jet colormap
    cmap = plt.get_cmap('jet')
    heatmap_rgb = cmap(heatmap)[..., :3]  # descartar alpha

    overlay = (1 - alpha) * img_rgb + alpha * heatmap_rgb
    return np.clip(overlay, 0, 1)


def visualize_gradcam_grid(model: nn.Module, dataset, device: torch.device,
                            n_positive: int = 3, n_negative: int = 3,
                            target_layer_name: str = "features.denseblock4",
                            output_path: Path | str | None = None) -> None:
    """
    Selecciona algunas imagenes del dataset, calcula Grad-CAM y las visualiza
    en un grid 2xN (fila 1: positivos, fila 2: negativos).
    """
    model.eval()
    target_layer = get_target_layer(model, target_layer_name)

    # Buscar indices de positivos y negativos en el dataset
    labels_series = dataset.df['label']
    pos_indices = labels_series[labels_series == 1].index.tolist()[:n_positive]
    neg_indices = labels_series[labels_series == 0].index.tolist()[:n_negative]

    n_cols = max(n_positive, n_negative)
    fig, axes = plt.subplots(2, n_cols, figsize=(4 * n_cols, 8))

    for row, (title, indices) in enumerate([
        ("Cardiomegaly (label=1)", pos_indices),
        ("No Finding (label=0)", neg_indices),
    ]):
        for col in range(n_cols):
            ax = axes[row, col] if n_cols > 1 else axes[row]
            if col >= len(indices):
                ax.axis('off')
                continue

            idx = indices[col]
            sample = dataset[idx]
            image_tensor = sample['image'].to(device)
            true_label = sample['label'].item()
            image_name = sample['image_index']

            # Prediccion del modelo
            with torch.no_grad():
                logits = model(image_tensor.unsqueeze(0))
                probs = torch.softmax(logits, dim=1)
                pred_prob = probs[0, 1].item()
                pred_label = int(pred_prob >= 0.5)

            # Grad-CAM para la clase verdadera
            heatmap = compute_gradcam_for_sample(
                model, image_tensor, target_class=true_label,
                target_layer=target_layer,
            )

            # Superponer
            img_np = image_tensor.squeeze().cpu().numpy()
            img_norm = normalize_for_display(img_np)
            overlay = overlay_heatmap(img_norm, heatmap, alpha=0.45)

            ax.imshow(overlay)
            correct = (pred_label == true_label)
            marker = "✓" if correct else "✗"
            color = 'green' if correct else 'red'
            ax.set_title(
                f"{image_name}\ntrue={true_label} | pred={pred_label} "
                f"(p={pred_prob:.2f}) {marker}",
                fontsize=9, color=color,
            )
            ax.axis('off')

        # Label de fila
        if n_cols > 0:
            axes[row, 0].set_ylabel(title, fontsize=12, rotation=90,
                                     labelpad=20, fontweight='bold')

    plt.suptitle("Grad-CAM sobre test set (colores calidos = regiones importantes)",
                 fontsize=13, y=1.00)
    plt.tight_layout()

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_path, dpi=120, bbox_inches='tight')
        print(f"Grad-CAM guardado en: {output_path}")

    plt.show()
'''
path = f'{CODE_DIR}/src/explainability/gradcam.py'
Path(path).parent.mkdir(parents=True, exist_ok=True)
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 3. Ejecutar evaluacion sobre TEST

Este es el momento de la verdad: metricas sobre datos que el modelo nunca vio.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/run_eval.py'],
    capture_output=True, text=True,
)
print('STDOUT:')
print(result.stdout)
print()
print('STDERR (log):')
print(result.stderr)


## 4. Cargar metricas de test y compararlas con val

In [ ]:
import json

test_metrics_path = f'{PROJECT_ROOT}/experiments/test_metrics.json'
history_path = f'{PROJECT_ROOT}/experiments/history.json'

with open(test_metrics_path) as f:
    test_data = json.load(f)
with open(history_path) as f:
    history_data = json.load(f)

test_m = test_data['metrics']
best_val_m = history_data['history'][history_data['best_epoch'] - 1]['val_metrics']

print('COMPARACION VAL vs TEST (mejor modelo):')
print('=' * 70)
print(f"{'Metric':<15} {'VAL':>10} {'TEST':>10} {'Delta':>10}")
print('-' * 70)
for key in ['auc', 'accuracy', 'precision', 'sensitivity', 'specificity']:
    v = best_val_m[key]
    t = test_m[key]
    delta = t - v
    arrow = '↓' if delta < -0.05 else ('↑' if delta > 0.05 else '≈')
    print(f"{key:<15} {v:>10.4f} {t:>10.4f} {delta:>+10.4f} {arrow}")

print()
print(f"Confusion matrix (test): TP={test_m['tp']} FP={test_m['fp']} "
      f"TN={test_m['tn']} FN={test_m['fn']}")
print(f"N de muestras: {test_m['n_samples']}")


## 5. Curva ROC sobre test

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

probs = np.array(test_data['probs'])
labels = np.array(test_data['labels'])

fpr, tpr, thresholds = roc_curve(labels, probs)
auc_val = test_m['auc']

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, linewidth=2, label=f'ROC (AUC={auc_val:.4f})', color='steelblue')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Azar (AUC=0.5)')

# Marcar el punto del threshold 0.5
sens = test_m['sensitivity']
spec = test_m['specificity']
ax.scatter([1 - spec], [sens], color='red', s=100, zorder=5,
           label=f'Threshold=0.5 (Sens={sens:.2f}, Spec={spec:.2f})')

ax.set_xlabel('1 - Especificidad (FPR)')
ax.set_ylabel('Sensibilidad (TPR)')
ax.set_title(f'Curva ROC sobre TEST (n={len(labels)})')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments/test_roc.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: experiments/test_roc.png')


## 6. Matriz de confusion

In [ ]:
import seaborn as sns

cm_array = np.array([[test_m['tn'], test_m['fp']],
                     [test_m['fn'], test_m['tp']]])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_array, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred No Finding', 'Pred Cardiomegaly'],
            yticklabels=['True No Finding', 'True Cardiomegaly'],
            cbar=False, square=True, annot_kws={'size': 14})
ax.set_title(f'Matriz de Confusion - TEST (threshold=0.5)')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments/test_confusion_matrix.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: experiments/test_confusion_matrix.png')


## 7. Grad-CAM sobre test

Cargamos el mejor modelo y generamos mapas de calor sobre 6 imagenes de test.
**Esperamos que en las cardiomegalias el mapa ilumine la region cardiaca.**

In [ ]:
# Limpiar cache de imports
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]

from src.training.utils import load_config, get_device
from src.datasets.nih import build_nih_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.evaluation.evaluate import load_model_from_checkpoint
from src.explainability.gradcam import visualize_gradcam_grid

cfg = load_config(f'{CODE_DIR}/configs/baseline.yaml')
device = get_device()

# Cargar mejor modelo
ckpt_path = f'{PROJECT_ROOT}/experiments/checkpoints/baseline_best.pt'
model = load_model_from_checkpoint(ckpt_path, cfg, device)
print(f'Modelo cargado desde: {ckpt_path}')
print(f'Parametros totales: {sum(p.numel() for p in model.parameters()):,}')

# Dataset de test (sin augmentation)
train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])
eval_tfm = build_eval_transform(image_size=cfg.preprocessing['image_size'])
_, _, test_ds = build_nih_datasets(cfg, train_tfm, eval_tfm)

# Generar Grad-CAM
visualize_gradcam_grid(
    model=model,
    dataset=test_ds,
    device=device,
    n_positive=3, n_negative=3,
    target_layer_name=cfg.gradcam['target_layer'],
    output_path=f'{PROJECT_ROOT}/experiments/gradcam_test.png',
)


## 8. Resumen final del Sprint 1

In [ ]:
print('=' * 70)
print('SPRINT 1 - BASELINE CXR BINARIO')
print('=' * 70)
print()
print(f'Tarea: Clasificacion binaria No Finding vs Cardiomegaly')
print(f'Dataset: NIH ChestX-ray14 (subset images_001)')
print(f'Modelo: DenseNet121 preentrenado en NIH, fine-tuning solo cabeza')
print(f'Splits: train=199 | val=40 | test=40 (estratificado por paciente)')
print()
print('METRICAS FINALES (sobre TEST, 40 imagenes nunca vistas):')
print(f'  AUC:         {test_m["auc"]:.4f}')
print(f'  Accuracy:    {test_m["accuracy"]:.4f}')
print(f'  Precision:   {test_m["precision"]:.4f}')
print(f'  Sensitivity: {test_m["sensitivity"]:.4f}')
print(f'  Specificity: {test_m["specificity"]:.4f}')
print()
print('Artefactos guardados en experiments/:')
print('  - history.json              (curvas de entrenamiento)')
print('  - test_metrics.json         (metricas finales)')
print('  - test_roc.png              (curva ROC)')
print('  - test_confusion_matrix.png (matriz de confusion)')
print('  - gradcam_test.png          (mapas de calor)')
print('  - checkpoints/baseline_best.pt (modelo entrenado, reproducible con seed=42)')
print()
print('=' * 70)
print('SPRINT 1 CERRADO')
print('=' * 70)


## 9. Proximos pasos (Sprint 2+)

Con el Sprint 1 cerrado, el roadmap natural es:

**Sprint 2: mejorar el baseline**
- Descargar `images_002.tar.gz` para mas datos.
- Probar `finetune_mode: "full"` con learning rate mas bajo (1e-5).
- Agregar LR scheduler (ReduceLROnPlateau).
- Cross-validation para reducir ruido en las metricas.

**Sprint 3: multiclass/multilabel**
- Extender a 3 clases (agregar Effusion).
- O pasar a multilabel con las 14 patologias NIH.

**Sprint 4: modelo hibrido CNN-ViT**
- Implementacion propia (no preentrenada).
- Comparar contra el baseline con los mismos splits.

**Sprint 5: interfaz funcional**
- Tal como plantea el Project Charter.
